In [8]:
import random
import os
from typing import List

from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate,
    Table,
    TableStyle,
    Paragraph,
    PageBreak
)
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.enums import TA_CENTER
from reportlab.lib import colors


# -------------------------------------------------
#  Random color
# -------------------------------------------------
def random_bingo_color():
    return colors.Color(
        red=random.uniform(0.2, 0.8),
        green=random.uniform(0.2, 0.8),
        blue=random.uniform(0.2, 0.8)
    )


# -------------------------------------------------
# Fit text into bingo cell
# -------------------------------------------------
def fit_paragraph(text, max_font=12, min_font=7):
    length = len(text)

    if length < 20:
        size = max_font
    elif length < 40:
        size = max_font - 1
    elif length < 60:
        size = max_font - 2
    elif length < 80:
        size = max_font - 3
    else:
        size = min_font

    style = ParagraphStyle(
        name="BingoCell",
        alignment=TA_CENTER,
        fontName="Helvetica",
        fontSize=size,
        leading=size + 2
    )

    return Paragraph(text, style)


In [13]:
# -------------------------------------------------
# Generate bingo cards
# -------------------------------------------------

import re
def prompt_mentions_name(prompt: str, name: str) -> bool:
    pattern = rf"\b{re.escape(name)}(?:['’]s?|)\b"
    return re.search(pattern, prompt, re.IGNORECASE) is not None
    ##return name.lower() in prompt.lower()

def generate_named_bingo_cards(
    prompts: List[str],
    names: List[str],
    grid_size: int = 5,
    free_space: bool = True
):
    needed = grid_size * grid_size - (1 if free_space else 0)
    if len(prompts) < needed:
        raise ValueError(f"You need at least {needed} prompts.")

    cards = []

    for name in names:
        ##selected = random.sample(prompts, needed)
        valid_prompts = [
            p for p in prompts
            if not prompt_mentions_name(p, name)
        ]

        if len(valid_prompts) < needed:
            raise ValueError(
                f"Not enough valid prompts for {name} "
                f"after excluding name-based prompts."
            )

        selected = random.sample(valid_prompts, needed)
        grid = []
        idx = 0

        for r in range(grid_size):
            row = []
            for c in range(grid_size):
                if free_space and r == c == grid_size // 2:
                    row.append(fit_paragraph("FREE"))
                else:
                    row.append(fit_paragraph(selected[idx]))
                    idx += 1
            grid.append(row)

        cards.append({
            "name": name,
            "grid": grid
        })

    return cards

In [14]:
# -------------------------------------------------
# Export to PDF
# -------------------------------------------------
def export_named_cards_to_pdf(named_cards, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    doc = SimpleDocTemplate(output_path, pagesize=A4)
    styles = getSampleStyleSheet()
    elements = []

    cell_size = 90

    for card in named_cards:
        theme_color = random_bingo_color()

        # Name header
        name_style = styles["Title"].clone("NameStyle")
        name_style.textColor = theme_color
        elements.append(Paragraph(card["name"], name_style))

        # Bingo grid
        table = Table(
            card["grid"],
            colWidths=[cell_size] * 5,
            rowHeights=[cell_size] * 5
        )

        table.setStyle(TableStyle([
            ("GRID", (0, 0), (-1, -1), 2, theme_color),
            ("ALIGN", (0, 0), (-1, -1), "CENTER"),
            ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
            ("BACKGROUND", (2, 2), (2, 2), colors.whitesmoke),
        ]))

        elements.append(table)
        elements.append(PageBreak())

    doc.build(elements)

In [15]:
if __name__ == "__main__":

    prompts = [
        "Someone says 'Align'",
        "Zack gets drunk and goes bed early",
        "Chinese team top 3",
        "SA team top 3",
        "NA team top 3",
        "West European team top 3",
        "East European team top 3",
        "Smoking group forms",
        "Team wins at crib by more than 30 points",
        "Adam breaks a toilet",
        "Jokes that liquid didnt qualify",
        "Meet Blitz/Quinn",
        "TeaGuv remembers us",
        "Someone mentions ESL 2018",
        "Someone makes a dog bark sound",
        "Rampage on mainstage",
        "<19 min game on mainstage",
        "Naga Carry picked on mainstage",
        "Someone shit talking Birmingham",
        "Peaky blinders reference at ESL",
        "Guinness runs out at the arena",
        "More than 1 rapier in a singular mainstage game",
        "XG finish 2nd",
        "Someone says 'Get jonned'",
        "Jon comes to ESL",
        "Luke comes to ESL",
        "Someone learns new, but obvious to the group dota mechanic",
        "Mcdonalds breakfast",
        "Fraser or Adam question pro BKB purchase",
        "Huskar 1st picked",
        "Someone says 'Polycule'",
        "Tamatanga evening meal",
        "Swedes ditch group for Pro friends",
        "Angus talks shit on popular opinion",
        "Jon/Henry says 'The wife...'",
        "Henry talks about Spa hotels",
        "Mention of Arjun's tinder account",
        "Robbie wears black trousers, top, jacket & hat",
        "Walk past pro in city centre",
        "Non-national in group bad mouthing England",
        "Daphne says 'Quinn is soo nice'",
        "Angus's brain hurts and misses a game",
        "Anyone plays emulator/osrs in main arena",
        "Someone says 'Even i could do that' when watching a game",
        "Fraser says he doesnt care about something",
        "Someone says 'Arjun nowhere to be seen'",
        "No bad mouthing of Robbie's dota performances at any point",
        "Gem lost within 1 minute of purchase on mainstage",
        "Someone says 'XP waste'",
        "Zack endangers someone (e.g. pushing onto road)",
        "Fraser says 'X Nigma player goated'",
        "Arjun wins werewolf type game",
        "Perudo; all players remaining with 1 die left",
        "Perudo: losing/lying when calling six sixes",
        "Perudo: lost dice",
        "Stolen aegis on mainstage",
        "Meepo on mainstage",
        "Angus displays dislike of Jenkins",
        "No stun pro draft",
        "Someone says 'Win condition'",
        "Caster says 'RTZ' when someone gets cliffed or manta's",
        "Kacper says 'Chinese Pasta'",
        "Adam/Kacper say 'Bomba'",
        "James mentions Mable",
        "James talks about the offlane being dead",
        "James mentions how he always buys auras and teamfight",
        "Anyone mentions lost prophets",
        "Jon says someone should 'uninstall'",
        "Drunk political argument",
        "Lager vs ale deabte",
        "Someone crashes out post 'discussion'",
        "Someone gets 'svaged'",
        "Someone says 'formerly maxed'",
        "Jolina says woke liberal nonsense",
        "Jon/Zack mention Corpus Christi",
        "Werewolves: 'The spartacus' e.g. More than 1 seer",
        "Jon does work instead of games in Birminham",
        "Bad mouthing of eastern europe",
        "Deliotte bad",
        "Recruit extra player e.g 'Birmingham Adam'",
        "Zack pick up woman",
        "Daphne talks about food delviery e.g VoltGate",
        "Daphne mentions vegetarianism",
        "Anyone mentions Brexit",
        "Zack brings capitalism into random conversation",
        "Complaints about zack or henry's snoring"

        




        





    
 
]

names = [
    "Angus",
    "Robbie",
    "Fraser",
    "Arjun",
    "Henry",
    "James",
    "Zack",
    "Kacper",
    "Adam",
    "Jolina",
    "Daphne",
    "Jon"
]




In [16]:
cards = generate_named_bingo_cards(prompts, names)

export_named_cards_to_pdf(
    cards,
    output_path=r"C:\Users\Angus\OneDrive\Documents\random stuff\Bingo\bingo_2026.pdf"
)

print("Bingo cards PDF created successfully!")

Bingo cards PDF created successfully!
